In [3]:
import pandas as pd 
import json
import numpy as np
import seaborn as sns

In [4]:
df=pd.read_json(r"C:\Users\nice\Desktop\Puresoft\Pures_Soft-Recommendation-System\Data\Users\orginal\user_events.json")

In [5]:
df

,events
0,"[{'user_id': 'ali', 'event_type': 'product_ope..."
1,"[{'user_id': 'ali', 'event_type': 'product_ope..."
2,"[{'user_id': 'ali', 'event_type': 'product_ope..."
3,"[{'user_id': 'ali', 'event_type': 'product_ope..."
4,"[{'user_id': 'ali', 'event_type': 'product_ope..."
...,...
245,"[{'user_id': 'user_002', 'event_type': 'produc..."
246,"[{'user_id': 'user_002', 'event_type': 'produc..."
247,"[{'user_id': 'user_003', 'event_type': 'produc..."
248,"[{'user_id': 'user_003', 'event_type': 'produc..."


In [6]:
df["events"][0]

[{'user_id': 'ali',
  'event_type': 'product_open',
  'timestamp': '2026-02-20T22:11:38.140966',
  'product_link': 'https://afaq-stores.com/product-details/576'}]

In [7]:
df.duplicated().sum()

0

In [8]:
df_expanded = df.explode("events").reset_index(drop=True)
events_df = pd.json_normalize(df_expanded["events"])
final_df = pd.concat([df_expanded.drop(columns=["events"]), events_df], axis=1)

In [9]:
df=final_df

In [10]:
df

,user_id,event_type,timestamp,product_link,duration,score
0,ali,product_open,2026-02-20T22:11:38.140966,https://afaq-stores.com/product-details/576,NaN,NaN
1,ali,product_open,2026-02-20T22:11:38.140966,https://afaq-stores.com/product-details/576,NaN,NaN
2,ali,add_to_cart,2026-02-20T22:11:39.649078,https://afaq-stores.com/product-details/576,NaN,NaN
3,ali,product_open,2026-02-20T22:11:38.140966,https://afaq-stores.com/product-details/576,NaN,NaN
4,ali,add_to_cart,2026-02-20T22:11:39.649078,https://afaq-stores.com/product-details/576,NaN,NaN
...,...,...,...,...,...,...
996,user_003,product_open,2026-02-25T10:26:40.270200,https://afaq-stores.com/product-details/9750,NaN,1.0
997,user_003,buy_click,2026-02-25T10:26:41.675261,https://afaq-stores.com/product-details/9750,NaN,5.0
998,user_003,product_open,2026-02-25T10:26:40.270200,https://afaq-stores.com/product-details/9750,NaN,1.0
999,user_003,buy_click,2026-02-25T10:26:41.675261,https://afaq-stores.com/product-details/9750,NaN,5.0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   user_id       1001 non-null   object 
 1   event_type    1001 non-null   object 
 2   timestamp     1001 non-null   object 
 3   product_link  1001 non-null   object 
 4   duration      290 non-null    float64
 5   score         869 non-null    float64
dtypes: float64(2), object(4)
memory usage: 47.1+ KB


In [12]:
df.drop(columns=["score"],inplace=True)

In [13]:
df["duration"]=df["duration"].fillna(0)

In [14]:
df.isna().sum()

user_id         0
event_type      0
timestamp       0
product_link    0
duration        0
dtype: int64

In [15]:
df["event_type"].unique()

array(['product_open', 'add_to_cart', 'product_exit', 'buy_click'],
      dtype=object)

In [16]:
event={
    "product_open":1,
    "add_to_cart":3,
    "buy_click":5,
    "product_exit":0
}

In [17]:
df["duration"].describe()

count    1001.000000
mean        1.519560
std         2.917007
min         0.000000
25%         0.000000
50%         0.000000
75%         2.840000
max        19.310000
Name: duration, dtype: float64

In [18]:
df["event_int"]=df["event_type"].map(event)

In [19]:
df["duration_norm"]=df["duration"]/df["duration"].max()

In [20]:
df["score"]=df["event_int"]+df["duration_norm"]

In [21]:
user_item_scores=df.groupby(["user_id","product_link"],as_index=False)["score"].sum()

In [22]:
df

,user_id,event_type,timestamp,product_link,duration,event_int,duration_norm,score
0,ali,product_open,2026-02-20T22:11:38.140966,https://afaq-stores.com/product-details/576,0.00,1,0.000000,1.000000
1,ali,product_open,2026-02-20T22:11:38.140966,https://afaq-stores.com/product-details/576,0.00,1,0.000000,1.000000
2,ali,add_to_cart,2026-02-20T22:11:39.649078,https://afaq-stores.com/product-details/576,0.00,3,0.000000,3.000000
3,ali,product_open,2026-02-20T22:11:38.140966,https://afaq-stores.com/product-details/576,0.00,1,0.000000,1.000000
4,ali,add_to_cart,2026-02-20T22:11:39.649078,https://afaq-stores.com/product-details/576,0.00,3,0.000000,3.000000
...,...,...,...,...,...,...,...,...
996,user_003,product_open,2026-02-25T10:26:40.270200,https://afaq-stores.com/product-details/9750,0.00,1,0.000000,1.000000
997,user_003,buy_click,2026-02-25T10:26:41.675261,https://afaq-stores.com/product-details/9750,0.00,5,0.000000,5.000000
998,user_003,product_open,2026-02-25T10:26:40.270200,https://afaq-stores.com/product-details/9750,0.00,1,0.000000,1.000000
999,user_003,buy_click,2026-02-25T10:26:41.675261,https://afaq-stores.com/product-details/9750,0.00,5,0.000000,5.000000


In [23]:
user_item_scores

,user_id,product_link,score
0,1,https://afaq-stores.com/product-details/11909,13.285344
1,1,https://afaq-stores.com/product-details/20049,49.970999
2,1,https://afaq-stores.com/product-details/25214,9.128949
3,1,https://afaq-stores.com/product-details/30168,9.153806
4,1,https://afaq-stores.com/product-details/9230,44.599171
5,2,https://afaq-stores.com/product-details/13358,41.005179
6,2,https://afaq-stores.com/product-details/9575,34.841015
7,ali,https://afaq-stores.com/product-details/20049,2.892284
8,ali,https://afaq-stores.com/product-details/25214,13.133610
9,ali,https://afaq-stores.com/product-details/576,57.321077


In [24]:
user_item_scores["score"].describe()

count     37.000000
mean      47.561395
std       64.751969
min        2.892284
25%       13.214915
50%       34.841015
75%       50.018643
max      373.594511
Name: score, dtype: float64